# Daily Challenge — Fine-tuning an LLM with LoRA

This notebook adapts `bigscience/bloomz-560m` to English quotes using a LoRA adapter. LoRA freezes the base model and learns small low-rank matrices, so only a small fraction of the parameters are trained and saved.

## 1. Install dependencies

Run this cell in Colab or another Python environment with internet access. Training a 560M-parameter model is much faster on a GPU; the supplied training configuration keeps CPU mode for compatibility with the exercise hint.

In [ ]:
# %pip install -q peft==0.4.0 datasets transformers accelerate torch
# !mkdir -p cache

## 2. Load the base model, tokenizer, and 10% dataset sample

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "bigscience/bloomz-560m"
cache_dir = "./cache"

tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=cache_dir)
# BLOOM has no separate padding token; using EOS makes batch padding safe.
tokenizer.pad_token = tokenizer.eos_token

foundation_model = AutoModelForCausalLM.from_pretrained(
    model_name, cache_dir=cache_dir
)
foundation_model.config.pad_token_id = tokenizer.pad_token_id

# Select a reproducible 10% sample of the training split.
raw_data = load_dataset("Abirate/english_quotes", split="train", cache_dir=cache_dir)
sample_size = max(1, int(0.10 * len(raw_data)))
data = raw_data.shuffle(seed=42).select(range(sample_size))
print(f"Using {len(data)} of {len(raw_data)} quotes ({len(data) / len(raw_data):.0%}).")
data[0]

## 3. Tokenize the quotes

Appending the end-of-sequence token tells the causal language model where each quote ends. `DataCollatorForLanguageModeling(mlm=False)` later creates the shifted labels used for next-token prediction.

In [ ]:
max_length = 128

def tokenize_quotes(samples):
    quotes = [quote.strip() + tokenizer.eos_token for quote in samples["quote"]]
    return tokenizer(quotes, truncation=True, max_length=max_length)

tokenized_data = data.map(
    tokenize_quotes, batched=True, remove_columns=data.column_names
)
# The five-example subset keeps the CPU exercise practical. Replace this with
# `tokenized_data` to train on the complete 10% sample on a GPU.
train_sample = tokenized_data.select(range(min(5, len(tokenized_data))))
display(train_sample)

## 4. Configure and apply LoRA

BLOOM combines its query, key, and value projections in modules named `query_key_value`. We use rank 1 as requested: only the LoRA matrices attached to that target are trainable.

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=1,
    lora_alpha=1,
    target_modules=["query_key_value"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

peft_model = get_peft_model(foundation_model, lora_config)
peft_model.print_trainable_parameters()

## 5. Train the LoRA adapter

The learning rate is higher than in full-model fine-tuning because only the newly added adapter weights are learned. Set `use_cpu=False` (and restart the runtime if necessary) when training on a GPU.

In [ ]:
import os
import transformers
from transformers import TrainingArguments, Trainer

output_directory = os.path.join("./cache/working", "peft_lab_outputs")
training_args = TrainingArguments(
    report_to="none",
    output_dir=output_directory,
    auto_find_batch_size=True,
    learning_rate=3e-2,
    num_train_epochs=1,
    use_cpu=True,
    logging_steps=1,
    save_strategy="no",
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_sample,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)
trainer.train()

## 6. Save and reload the adapter

`save_pretrained` stores the small adapter configuration and learned weights, not another full 560M base-model copy. Reloading attaches those weights to a fresh copy of the base model.

In [ ]:
import time
from peft import PeftModel

time_now = time.strftime("%Y%m%d_%H%M%S")
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")
trainer.model.save_pretrained(peft_model_path)
print(f"Saved LoRA adapter to: {peft_model_path}")

# A fresh base model proves that the adapter can be reused independently.
inference_base_model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir=cache_dir)
inference_base_model.config.pad_token_id = tokenizer.pad_token_id
loaded_peft_model = PeftModel.from_pretrained(
    inference_base_model, peft_model_path, is_trainable=False
)
loaded_peft_model.eval()

## 7. Generate text with the fine-tuned model

In [ ]:
prompt = "Two things are infinite: "
device = next(loaded_peft_model.parameters()).device
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with __import__("torch").no_grad():
    outputs = loaded_peft_model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

## Conclusion

LoRA made it possible to specialize a large causal model while training and saving only adapter parameters. The base model remains reusable, and different adapters can be stored and swapped for different domains or tasks. Generated text will vary because sampling is enabled and the training subset is deliberately very small.